In [ ]:
from akd._base import ThinkingEvent, StreamingTokenEvent, CompletedEvent, ToolCallingEvent, ToolResultEvent, RunContext, HumanInputRequiredEvent, HumanResponseEvent
from akd._base.structures import RunUsage
from akd_ext.agents import EIEAgent, EIEAgentInputSchema, EIEAgentConfig
import os
import pandas as pd
from IPython.display import display


os.environ["OPENAI_API_KEY"] = ""
os.environ["EIE_MCP_KEY"] = ""

# Pricing per token (per 1M rates / 1_000_000)
MODEL_PRICING = {
    "gpt-5.2": {
        "input": 1.75 / 1_000_000,
        "cached_input": 0.175 / 1_000_000,
        "output": 14.00 / 1_000_000,
    },
    # "gpt-4o": {
    #     "input": 2.50 / 1_000_000,
    #     "cached_input": 1.25 / 1_000_000,
    #     "output": 10.00 / 1_000_000,
    # },
    # "gpt-5-nano": {
    #     "input": 0.05 / 1_000_000,
    #     "cached_input": 0.005 / 1_000_000,
    #     "output": 0.40 / 1_000_000,
    # },
    "gpt-5-mini": {
        "input": 0.25 / 1_000_000,
        "cached_input": 0.025 / 1_000_000,
        "output": 2.00 / 1_000_000,
    },
    # "gpt-4o-mini": {
    #     "input": 0.15 / 1_000_000,
    #     "cached_input": 0.075 / 1_000_000,
    #     "output": 0.60 / 1_000_000,
    # },
}

# Cumulative cost tracker — persists across cell re-runs
cost_log = []
_prev_usage = RunUsage()

def track_usage(run_context, step_label="", model="gpt-5.2"):
    """Track per-step and cumulative cost across all models."""
    global _prev_usage
    total = run_context.usage

    # Delta = current cumulative - previous cumulative
    delta_input = total.input_tokens - _prev_usage.input_tokens
    delta_output = total.output_tokens - _prev_usage.output_tokens
    delta_requests = total.requests - _prev_usage.requests
    delta_cached = (
        total.details.get("input_tokens_details.cached_tokens", 0)
        - _prev_usage.details.get("input_tokens_details.cached_tokens", 0)
    )
    delta_non_cached = delta_input - delta_cached

    row = {
        "step": len(cost_log) + 1,
        "label": step_label,
        "model": model,
        "llm_calls": delta_requests,
        "input_tokens": delta_input,
        "cached": delta_cached,
        "output_tokens": delta_output,
    }
    # Compute cost for every model from the same token counts
    for m, p in MODEL_PRICING.items():
        row[m] = (
            delta_non_cached * p["input"]
            + delta_cached * p["cached_input"]
            + delta_output * p["output"]
        )

    cost_log.append(row)

    _prev_usage = RunUsage(
        input_tokens=total.input_tokens,
        output_tokens=total.output_tokens,
        requests=total.requests,
        details=dict(total.details),
    )

    df = pd.DataFrame(cost_log)
    model_cols = list(MODEL_PRICING.keys())
    # Add totals row
    sum_cols = ["llm_calls", "input_tokens", "cached", "output_tokens"] + model_cols
    totals = df[sum_cols].sum()
    totals["step"] = ""
    totals["label"] = "TOTAL"
    totals["model"] = ""
    df = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)
    # Format money columns
    for col in model_cols:
        df[col] = df[col].map("${:.5f}".format)
    return df


In [ ]:
#Initialize the agent
agent = EIEAgent()
# run_context is like the memory/conversation history of the agent. Since the agent is just initialized, run_context should be null
run_context = None

In [ ]:
# Re-run this cell with query="your answer" to continue the conversation
query = "do the same for delhi" #Show me NO2 air quality data for Houston TX from January to June 2023 | yes | yes | 1 | yes

if run_context:
    run_context = RunContext.model_validate(run_context)
async for event in agent.astream(
    EIEAgentInputSchema(query=query),
    run_context=run_context,
):
    if isinstance(event, ToolCallingEvent):
        print("TOOL_CALL: ", event.data.tool_call.tool_name)
    if isinstance(event, ToolResultEvent):
        content = str(event.data.result.content)
        print("TOOL_RESULT: ", content[:200] + "..." if len(content) > 200 else content)
    if isinstance(event, ThinkingEvent):
        print(event.data.thinking_content, end="")
    if isinstance(event, StreamingTokenEvent):
        print(event.data.token, end="")
    if isinstance(event, HumanInputRequiredEvent):
        print(f"\n--- INTERRUPT: {event.data.human_input.question}")
    if isinstance(event, CompletedEvent):
        print("\n--- COMPLETED ---")

    run_context = event.run_context

df = track_usage(run_context, step_label=query[:50], model=agent.config.model_name)
display(df)

In [ ]:
# Inspect the run_context
run_context.model_dump()

## Testing with gpt-5-mini
The goal of this experiment with gpt-5-mini is to figure out if it can reliably follow the tool sequence and pass parameters correctly, for a tool-heavy agent like EIE

In [ ]:
#Initialize the agent with a smaller model
agent = EIEAgent(EIEAgentConfig(model_name='gpt-5-mini', reasoning_effort="low"))
# run_context is like the memory/conversation history of the agent. Since the agent is just initialized, run_context should be null
run_context = None


In [ ]:
agent.config.model_name

In [ ]:
# Re-run this cell with query="your answer" to continue the conversation
query = "yes"

if run_context:
    run_context = RunContext.model_validate(run_context)
async for event in agent.astream(
    EIEAgentInputSchema(query=query),
    run_context=run_context,
):
    if isinstance(event, ToolCallingEvent):
        print("TOOL_CALL: ", event.data.tool_call.tool_name)
    if isinstance(event, ToolResultEvent):
        content = str(event.data.result.content)
        print("TOOL_RESULT: ", content[:200] + "..." if len(content) > 200 else content)
    if isinstance(event, ThinkingEvent):
        print(event.data.thinking_content, end="")
    if isinstance(event, StreamingTokenEvent):
        print(event.data.token, end="")
    if isinstance(event, HumanInputRequiredEvent):
        print(f"\n--- INTERRUPT: {event.data.human_input.question}")
    if isinstance(event, CompletedEvent):
        print("\n--- COMPLETED ---")

    run_context = event.run_context

#df_mini = track_usage(run_context, step_label=query[:50], model=agent.config.model_name)

In [ ]:
## Result for gpt-5-mini : Good

## Testing with gpt-5-nano
The goal of this experiment with gpt-5-nano is to figure out if it can reliably follow the tool sequence and pass parameters correctly, for a tool-heavy agent like EIE

In [ ]:
#Initialize the agent with a smaller model
agent = EIEAgent(EIEAgentConfig(model_name='gpt-5-nano', reasoning_effort="low"))
# run_context is like the memory/conversation history of the agent. Since the agent is just initialized, run_context should be null
run_context = None


In [ ]:
# Re-run this cell with query="your answer" to continue the conversation
query = "yes"

if run_context:
    run_context = RunContext.model_validate(run_context)
async for event in agent.astream(
    EIEAgentInputSchema(query=query),
    run_context=run_context,
):
    if isinstance(event, ToolCallingEvent):
        print("TOOL_CALL: ", event.data.tool_call.tool_name)
    if isinstance(event, ToolResultEvent):
        content = str(event.data.result.content)
        print("TOOL_RESULT: ", content[:200] + "..." if len(content) > 200 else content)
    if isinstance(event, ThinkingEvent):
        print(event.data.thinking_content, end="")
    if isinstance(event, StreamingTokenEvent):
        print(event.data.token, end="")
    if isinstance(event, HumanInputRequiredEvent):
        print(f"\n--- INTERRUPT: {event.data.human_input.question}")
    if isinstance(event, CompletedEvent):
        print("\n--- COMPLETED ---")

    run_context = event.run_context

#df_mini = track_usage(run_context, step_label=query[:50], model=agent.config.model_name)

In [ ]:
## Result for gpt-5-nano : Okish but Weird